# Evaluation for BPI Challenge 2013 (open problems) models

Inside the folder `<project_root>/runs/bpic2013_open_problems` we have a list of folders named as `<percentage>%`, where `<percentage>` is the percentage of the dataset used for training the model.
In each of these folders we have a folder named as the best model found during the training phase based on the accuracy value.
Inside each of these folders we have the following files:
- `constraints_satisfaction_rate.csv`: a CSV file containing the constraints satisfaction rate for each of the test traces.
- `constraints_satisfactions.csv`: a CSV file containing the constraints satisfaction for each of the test traces.
- `predicted_traces.txt`: a TXT file containing the traces generated by the model for each of the test traces.
- `predictions.csv`: a CSV file containing the predictions step by step for each of the test traces.
- `results.json`: a JSON file containing the results of the evaluation of the model on the test set.

In [5]:
DATASET_NAME = "bpic2013_open_problems"

In [ ]:
from collections import namedtuple
import pathlib

project_root = pathlib.Path("../../..").parent.resolve()

Info = namedtuple("Info", ["model_args", "model_path", "results_path"])

models_path: dict[int, list[Info]] = {}

for dataset_percentage in range(20, 101, 20):
    checkpoints = [
        path
        for path in (project_root / "runs" / DATASET_NAME).rglob(
            f"{dataset_percentage}%/**/*.best_val_acc.pth"
        )
    ]
    results = [
        pathlib.Path(str(checkpoint).removesuffix(".pth"))
        / "step_by_step"
        / "results.json"
        for checkpoint in checkpoints
    ]
    args = [checkpoint.parent / "args.json" for checkpoint in checkpoints]
    models_path[dataset_percentage] = [
        Info(model_args=args, model_path=checkpoint, results_path=result)
        for args, checkpoint, result in zip(args, checkpoints, results)
    ]

## Comparison

In [7]:
import json
import pandas as pd

pd.set_option("max_colwidth", 400)

dataframes = {}

for percentage in models_path:
    dataframes[percentage] = pd.DataFrame(
        columns=[
            "lr",
            "dropout",
            "loss",
            "acc",
            "dld",
            "norm_dld",
            "constraints",
            "constraints_multiplier",
        ]
    )
    for info in models_path[percentage]:
        with open(info.model_args) as f:
            args = json.load(f)
        try:
            with open(info.results_path) as f:
                results = json.load(f)
                dataframes[percentage].loc[info.model_path.parent.name] = [
                    args["learning_rate"],
                    args["model"]["dropout"],
                    results["loss"],
                    results["acc"],
                    results["dld"],
                    results["norm_dld"],
                    args.get("constraints", None),
                    args.get("constraints_multiplier", None),
                ]
        except FileNotFoundError:
            print(f"Missing results for {info.model_path}")

/tmp/ipykernel_2522785/703172264.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dataframes[percentage].loc[info.model_path.parent.name] = [


In [8]:
for percentage in range(20, 101, 20):
    print("=" * 10 + f" {percentage}% " + "=" * 10)
    display(dataframes[percentage].sort_values("dld", ascending=True))
    print()

========== 20% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier



========== 40% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier



========== 60% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier



========== 80% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier



========== 100% ==========


,lr,dropout,loss,acc,dld,norm_dld,constraints,constraints_multiplier
20250319.1400.no_constraint,0.0001,0.2,0.298470,0.615385,4.76,0.427269,[],NaN
20250319.1403.constraints,0.0001,0.2,0.299580,0.608974,4.80,0.429769,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.001
20250319.1405.constraints,0.0001,0.2,0.298532,0.608974,4.80,0.429769,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.010
20250319.1414.constraints,0.0001,0.2,0.299548,0.608974,4.80,0.429769,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.001
20250319.1426.constraints,0.0001,0.2,0.299326,0.608974,4.80,0.429769,[Absence3[Accepted]*0.76],0.001
20250319.1416.constraints,0.0001,0.2,0.298853,0.605769,4.88,0.428466,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.010
20250319.1427.constraints,0.0001,0.2,0.298252,0.602564,4.92,0.436466,[Absence3[Accepted]*0.76],0.010
20250319.1418.constraints,0.0001,0.2,0.369915,0.557692,5.40,0.449457,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.100
20250319.1407.constraints,0.0001,0.2,0.400316,0.528846,5.76,0.448295,"[Absence2[Completed]*0.98, Init[Accepted]*0.92...",0.100
20250319.1429.constraints,0.0001,0.2,0.337581,0.525641,5.92,0.449448,[Absence3[Accepted]*0.76],0.100
